In [32]:
import pandas as pd
import numpy as np
np.random.seed(42)
path = r"C:\univ\Data mining\Project\data\vanilla.json"

df = pd.read_json(path)

print(df.head())
print(df.info())

       business_name sector  followers_count   post_date  posting_hour  \
0  Vanilla Palestine   Cafe           139000  2025-02-05           NaN   
1  Vanilla Palestine   Cafe           139000  2025-02-04           NaN   
2  Vanilla Palestine   Cafe           139000  2025-02-01           NaN   
3  Vanilla Palestine   Cafe           139000  2025-01-24           NaN   
4  Vanilla Palestine   Cafe           139000  2023-10-03           NaN   

  day_of_week     month post_type  \
0   Wednesday  February      Reel   
1     Tuesday  February      Reel   
2    Saturday  February      Reel   
3      Friday   January      Reel   
4     Tuesday   October      Reel   

                                        caption_text  caption_length  ...  \
0  شاركونا إجابتكم بالتعليقات 👇 #فانيلا_اليوم_وكل...              83  ...   
1  What’s your favorite cake ❤️🍰? #VANILLA_Today_...              65  ...   
2  كل يوم له بداية… وأجمل بداية بطاقة إيجابية #فا...              72  ...   
3  ومن أهم فوائدها إنها 

In [33]:
df.isna().sum()/len(df)

business_name           0.000000
sector                  0.000000
followers_count         0.000000
post_date               0.000000
posting_hour            1.000000
day_of_week             0.000000
month                   0.000000
post_type               0.000000
caption_text            0.000000
caption_length          0.000000
hashtags_count          0.000000
emoji_count             0.000000
likes_count             0.000000
comments_count          0.024194
views_count             0.000000
language                0.000000
CTA_present             0.000000
promo_post              0.000000
discount_percent        0.806452
mentions_location       0.000000
religious_theme         0.000000
patriotic_theme         0.000000
arabic_dialect_style    0.048387
sponsored               0.000000
dtype: float64

In [34]:
df = df.drop(columns=[ 'posting_hour', 'discount_percent'])

In [35]:
def print_missing():
    missing_ratio = df.isna().sum() / len(df)

    # Get columns where missing percentage > 0
    missing_columns = missing_ratio[missing_ratio > 0]

    print(missing_columns)

In [36]:
print_missing()

comments_count          0.024194
arabic_dialect_style    0.048387
dtype: float64


### `handle sponsored missing`

In [37]:
df.loc[:, 'sponsored'] = df['sponsored'].fillna(0)
df['sponsored'].value_counts()


sponsored
0    123
1      1
Name: count, dtype: int64

### `EDA`

In [38]:
# Keep only columns that have at least one missing value in the whole dataframe
missing_cols = df.columns[df.isna().any()]

# Percentage of missing values grouped by post_type
missing_by_post_type = (
    df.groupby("post_type")[missing_cols]
      .apply(lambda x: x.isna().sum() / len(x))
)

print(missing_by_post_type)

           comments_count  arabic_dialect_style
post_type                                      
Reel             0.000000              0.000000
post             0.000000              0.000000
reel             0.047619              0.095238


In [39]:
df.columns

Index(['business_name', 'sector', 'followers_count', 'post_date',
       'day_of_week', 'month', 'post_type', 'caption_text', 'caption_length',
       'hashtags_count', 'emoji_count', 'likes_count', 'comments_count',
       'views_count', 'language', 'CTA_present', 'promo_post',
       'mentions_location', 'religious_theme', 'patriotic_theme',
       'arabic_dialect_style', 'sponsored'],
      dtype='object')

In [40]:
def print_inconsistencies():
       categ_cols = set(df.columns) - set(['business_name', 'followers_count', 'post_date','caption_text', 'caption_length',
              'hashtags_count', 'emoji_count', 'likes_count', 'comments_count',
              'views_count',])
       for col in categ_cols:
              print(f"{col}: {df[col].nunique()}")
              print(f"{col}: {df[col].unique()}")

In [41]:
print_inconsistencies()

CTA_present: 2
CTA_present: [ True False]
post_type: 3
post_type: ['Reel' 'reel' 'post']
sector: 3
sector: ['Cafe' 'influancers' 'Cafes/Restaurants']
patriotic_theme: 2
patriotic_theme: [False  True]
arabic_dialect_style: 4
arabic_dialect_style: [True False 'Levantine' 'Palestinian' None]
day_of_week: 7
day_of_week: ['Wednesday' 'Tuesday' 'Saturday' 'Friday' 'Sunday' 'Monday' 'Thursday']
mentions_location: 2
mentions_location: [False  True]
language: 4
language: ['Arabic' 'English' 'Mixed' 'Arabic-English']
promo_post: 2
promo_post: [False  True]
month: 11
month: ['February' 'January' 'October' 'September' 'August' 'May' 'April' 'March'
 'December' 'November' 'June']
religious_theme: 2
religious_theme: [False  True]
sponsored: 2
sponsored: [0 1]


### `Handle Language inconsistency`

In [42]:
df['language'].unique()

array(['Arabic', 'English', 'Mixed', 'Arabic-English'], dtype=object)

In [43]:
mixed_mask = df['language'].isin([
    'mixed',
    'Arabic-English',
    'Mixed',
    'Arabic/English Mixed',
    'English/Arabic Mixed'
])

df.loc[mixed_mask, 'language'] = 'Mixed'

In [44]:
df['language'].unique()

array(['Arabic', 'English', 'Mixed'], dtype=object)

In [45]:
df['language'].isna().sum()

np.int64(0)

`Language inconsistency handled`

### `dailectal arabic`

In [46]:
df['arabic_dialect_style'].isna().sum()

np.int64(6)

In [47]:
df['arabic_dialect_style'].value_counts()

arabic_dialect_style
True           71
False          29
Palestinian    17
Levantine       1
Name: count, dtype: int64

In [48]:
levantine_mask = ~df['arabic_dialect_style'].isin([True, False])

In [49]:
df.loc[levantine_mask, 'arabic_dialect_style'] = True

In [50]:
df[~df['arabic_dialect_style'].isin([True, False])]

,business_name,sector,followers_count,post_date,day_of_week,month,post_type,caption_text,caption_length,hashtags_count,...,comments_count,views_count,language,CTA_present,promo_post,mentions_location,religious_theme,patriotic_theme,arabic_dialect_style,sponsored


In [51]:
df['arabic_dialect_style'].fillna(True)

C:\Users\LEGION\AppData\Local\Temp\ipykernel_20732\3862925752.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['arabic_dialect_style'].fillna(True)


0       True
1      False
2       True
3       True
4       True
       ...  
119     True
120    False
121     True
122     True
123     True
Name: arabic_dialect_style, Length: 124, dtype: bool

`dialectal hadneld`

`Check for consistencies again`

In [52]:
print_inconsistencies()

CTA_present: 2
CTA_present: [ True False]
post_type: 3
post_type: ['Reel' 'reel' 'post']
sector: 3
sector: ['Cafe' 'influancers' 'Cafes/Restaurants']
patriotic_theme: 2
patriotic_theme: [False  True]
arabic_dialect_style: 2
arabic_dialect_style: [True False]
day_of_week: 7
day_of_week: ['Wednesday' 'Tuesday' 'Saturday' 'Friday' 'Sunday' 'Monday' 'Thursday']
mentions_location: 2
mentions_location: [False  True]
language: 3
language: ['Arabic' 'English' 'Mixed']
promo_post: 2
promo_post: [False  True]
month: 11
month: ['February' 'January' 'October' 'September' 'August' 'May' 'April' 'March'
 'December' 'November' 'June']
religious_theme: 2
religious_theme: [False  True]
sponsored: 2
sponsored: [0 1]


In [53]:
print_missing()

comments_count    0.024194
dtype: float64


In [54]:
# Normalize post_type values

image_mask = df['post_type'] == 'Image'
reel_mask = df['post_type'] == 'Reel'

df.loc[image_mask, 'post_type'] = 'post'
df.loc[reel_mask, 'post_type'] = 'reel'

print(df['post_type'].unique())

['reel' 'post']



`Image and reel consistencies ahdnled`

In [55]:
print_missing()

comments_count    0.024194
dtype: float64


`Propogated`

`Fill likes and comments with mean/median form the same page`

In [56]:
cols = [ 'comments_count']

for col in cols:
    df[col] = df[col].fillna(
        df.groupby('business_name')[col].transform('median')
    )

In [57]:
print_missing()

Series([], dtype: float64)


In [58]:
df.loc[:, 'sector'] = 'Cafes/Restaurants'

In [59]:
print_inconsistencies()

CTA_present: 2
CTA_present: [ True False]
post_type: 2
post_type: ['reel' 'post']
sector: 1
sector: ['Cafes/Restaurants']
patriotic_theme: 2
patriotic_theme: [False  True]
arabic_dialect_style: 2
arabic_dialect_style: [True False]
day_of_week: 7
day_of_week: ['Wednesday' 'Tuesday' 'Saturday' 'Friday' 'Sunday' 'Monday' 'Thursday']
mentions_location: 2
mentions_location: [False  True]
language: 3
language: ['Arabic' 'English' 'Mixed']
promo_post: 2
promo_post: [False  True]
month: 11
month: ['February' 'January' 'October' 'September' 'August' 'May' 'April' 'March'
 'December' 'November' 'June']
religious_theme: 2
religious_theme: [False  True]
sponsored: 2
sponsored: [0 1]


In [60]:
# Save the processed DataFrame inside ../data
df.to_json(
    "../data/vanilla_processed.json",
    orient="records",
    force_ascii=False,
    indent=4
)

print("Saved as ../data/vanilla_processed.json")

Saved as ../data/vanilla_processed.json
